# Pregunta 3 — CFA, SEM e invariancia multigrupo con lavaan

Este notebook resuelve la Pregunta 3 usando `lavaan`, manteniendo Python solo como entorno de ejecución. La base se descarga desde el repositorio si no está disponible localmente.


## Enunciado textual

Como recién llegado a la consultora, se analiza un proyecto sobre condiciones de trabajo, satisfacción y permanencia de trabajadores del sector telecomunicaciones. Los constructos son actitud hacia compañeros de trabajo, percepción del ambiente de trabajo, satisfacción con el trabajo, compromiso organizacional e intención de permanecer en el trabajo. Cada constructo se mide con cuatro indicadores observados. Se solicita evaluar: (a) la calidad del modelo de medición, incluyendo confiabilidad y validez; (b) cuál modelo estructural representa mejor los datos; (c) si las relaciones son equivalentes entre hombres y mujeres; y (d) interpretar los resultados.


## Respuesta

Se estima primero un CFA de cinco factores. Luego se comparan el modelo completamente mediado y el parcialmente mediado. Finalmente se evalúa invariancia multigrupo por género.




### Figuras del enunciado

**Constructos e ítems considerados en la Pregunta 3**

![Tabla de constructos](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_tabla_constructos.png?raw=1)

**Modelo completamente mediado**

![Modelo completamente mediado](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_modelo_completamente_mediado.png?raw=1)

**Modelo parcialmente mediado**

![Modelo parcialmente mediado](https://github.com/sebabecerra/Metodos-Cuantitativos-DEN/blob/main/Tarea2/assets/p3_modelo_parcialmente_mediado.png?raw=1)


In [3]:
# Instalación condicional para Google Colab.
# En local no hace nada si R y los paquetes ya existen.
import shutil, subprocess, textwrap, os, sys
if shutil.which('Rscript') is None:
    subprocess.check_call(['apt-get', 'update', '-qq'])
    subprocess.check_call(['apt-get', 'install', '-y', 'r-base'])
subprocess.check_call(['Rscript', '-e', "pkgs <- c('lavaan','haven','psych'); missing <- pkgs[!sapply(pkgs, requireNamespace, quietly=TRUE)]; if(length(missing)>0) install.packages(missing, repos='https://cloud.r-project.org')"])
print('R/lavaan listo')


R/lavaan listo


In [2]:
from pathlib import Path
r_code = r'''
# Workaround for macOS/R setups where parallel::detectCores() returns NA.
ns <- asNamespace('parallel')
unlockBinding('detectCores', ns)
assign('detectCores', function(all.tests=FALSE, logical=TRUE) 8L, envir=ns)
lockBinding('detectCores', ns)

pkgs <- c('lavaan','haven','psych')
missing <- pkgs[!sapply(pkgs, requireNamespace, quietly=TRUE)]
if (length(missing) > 0) install.packages(missing, repos='https://cloud.r-project.org')

library(lavaan)
library(haven)
library(psych)

find_data <- function() {
  candidates <- c(
    '../data/Laboral.sav',
    '../../data/Laboral.sav',
    'Tarea2/data/Laboral.sav',
    'data/Laboral.sav',
    'Laboral.sav'
  )
  for (f in candidates) if (file.exists(f)) return(f)
  url <- 'https://raw.githubusercontent.com/sebabecerra/Metodos-Cuantitativos-DEN/main/Tarea2/data/Laboral.sav'
  download.file(url, 'Laboral.sav', mode='wb', quiet=TRUE)
  return('Laboral.sav')
}

base <- find_data()
d <- as.data.frame(read_sav(base))
vars <- c(paste0('AC',1:4), paste0('PA',1:4), paste0('ST',1:4), paste0('CO',1:4), paste0('IP',1:4), 'GENERO')
d <- d[, vars]
d$GENERO <- as.integer(d$GENERO)

cat('CHEQUEO INICIAL DE LA BASE\n')
cat('Archivo usado:', base, '\n')
cat('Número de observaciones:', nrow(d), '\n')
cat('Número de variables analizadas:', ncol(d), '\n')
cat('Frecuencia GENERO: 0=Masculino, 1=Femenino\n')
print(table(d$GENERO, useNA='ifany'))
cat('Datos perdidos totales:', sum(is.na(d)), '\n')
cat('Datos perdidos por variable:\n')
print(colSums(is.na(d)))
cat('Descriptivos básicos de los indicadores y género:\n')
print(round(psych::describe(d)[, c('n','mean','sd','min','max')], 3))
cat('\n')

measurement <- '
AC =~ AC1 + AC2 + AC3 + AC4
PA =~ PA1 + PA2 + PA3 + PA4
ST =~ ST1 + ST2 + ST3 + ST4
CO =~ CO1 + CO2 + CO3 + CO4
IP =~ IP1 + IP2 + IP3 + IP4
'
complete_model <- paste0(measurement, '
ST ~ PA + AC
CO ~ PA + AC + ST
IP ~ ST + CO
')
partial_model <- paste0(measurement, '
ST ~ PA + AC
CO ~ PA + AC + ST
IP ~ PA + AC + ST + CO
')

fit_cfa <- cfa(measurement, data=d, estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_complete <- sem(complete_model, data=d, estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_partial <- sem(partial_model, data=d, estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)

fm <- c('chisq.scaled','df.scaled','pvalue.scaled','cfi.robust','tli.robust','rmsea.robust','srmr','aic','bic')
cat('1) CFA robusto\n')
print(round(fitMeasures(fit_cfa, fm), 4))

pe_cfa <- parameterEstimates(fit_cfa, standardized=TRUE, ci=TRUE)
loadings <- pe_cfa[pe_cfa$op=='=~', c('lhs','rhs','est','se','z','pvalue','ci.lower','ci.upper','std.all')]
cat('\n2) Cargas factoriales CFA\n')
print(loadings)

cat('\n3) Confiabilidad y AVE\n')
quality <- data.frame()
for (fac in c('AC','PA','ST','CO','IP')) {
  items <- pe_cfa[pe_cfa$op=='=~' & pe_cfa$lhs==fac,]
  lambda <- items$std.all
  cr <- sum(lambda)^2 / (sum(lambda)^2 + sum(1-lambda^2))
  ave <- mean(lambda^2)
  alpha <- psych::alpha(d[, items$rhs], warnings=FALSE)$total$raw_alpha
  quality <- rbind(quality, data.frame(Constructo=fac, Alfa=alpha, CR=cr, AVE=ave, sqrtAVE=sqrt(ave), CargaMin=min(lambda), CargaMax=max(lambda)))
}
print(round_df <- within(quality, {Alfa=round(Alfa,4); CR=round(CR,4); AVE=round(AVE,4); sqrtAVE=round(sqrtAVE,4); CargaMin=round(CargaMin,4); CargaMax=round(CargaMax,4)}))

cat('\n4) Correlaciones latentes CFA y Fornell-Larcker\n')
std <- standardizedSolution(fit_cfa)
latcorr <- std[std$op=='~~' & std$lhs %in% c('AC','PA','ST','CO','IP') & std$rhs %in% c('AC','PA','ST','CO','IP') & std$lhs != std$rhs, c('lhs','rhs','est.std')]
print(latcorr)
cat('Correlación latente máxima absoluta:', max(abs(latcorr$est.std)), '\n')

htmt <- function(data, a, b) {
  ra <- cor(data[,a], use='pairwise.complete.obs')
  rb <- cor(data[,b], use='pairwise.complete.obs')
  rab <- cor(data[,a], data[,b], use='pairwise.complete.obs')
  mean(abs(as.vector(rab))) / sqrt(mean(abs(ra[upper.tri(ra)])) * mean(abs(rb[upper.tri(rb)])))
}
constructos <- list(AC=paste0('AC',1:4), PA=paste0('PA',1:4), ST=paste0('ST',1:4), CO=paste0('CO',1:4), IP=paste0('IP',1:4))
htmt_vals <- c()
for (i in seq_along(constructos)) for (j in seq_along(constructos)) if (i>j) htmt_vals <- c(htmt_vals, htmt(d, constructos[[i]], constructos[[j]]))
cat('HTMT máximo:', max(htmt_vals), '\n')

cat('\n5) Comparación SEM completo vs parcial\n')
fits_table <- rbind(Completo=fitMeasures(fit_complete, fm), Parcial=fitMeasures(fit_partial, fm))
print(round(fits_table, 4))
cat('\nPrueba robusta de diferencia Satorra-Bentler\n')
print(lavTestLRT(fit_complete, fit_partial, method='satorra.bentler.2010'))

cat('\n6) Coeficientes estructurales modelo parcial\n')
pe_partial <- parameterEstimates(fit_partial, standardized=TRUE, ci=TRUE)
print(pe_partial[pe_partial$op=='~', c('lhs','rhs','est','se','z','pvalue','ci.lower','ci.upper','std.all')])

cat('\n7) Efectos indirectos bootstrap (ML, 5000 remuestras)\n')
model_boot <- '
AC =~ AC1 + AC2 + AC3 + AC4
PA =~ PA1 + PA2 + PA3 + PA4
ST =~ ST1 + ST2 + ST3 + ST4
CO =~ CO1 + CO2 + CO3 + CO4
IP =~ IP1 + IP2 + IP3 + IP4
ST ~ a1*PA + a2*AC
CO ~ d1*PA + d2*AC + a3*ST
IP ~ c1*PA + c2*AC + b1*ST + b2*CO
pa_st_ip := a1*b1
pa_co_ip := d1*b2
pa_st_co_ip := a1*a3*b2
pa_ind_total := pa_st_ip + pa_co_ip + pa_st_co_ip
ac_st_ip := a2*b1
ac_co_ip := d2*b2
ac_st_co_ip := a2*a3*b2
ac_ind_total := ac_st_ip + ac_co_ip + ac_st_co_ip
'
set.seed(12345)
fit_boot <- sem(model_boot, data=d, estimator='ML', missing='fiml', std.lv=TRUE, se='bootstrap', bootstrap=5000, ncpus=1)
pe_boot <- parameterEstimates(fit_boot, standardized=TRUE, ci=TRUE, boot.ci.type='perc')
print(pe_boot[pe_boot$op==':=', c('lhs','est','se','z','pvalue','ci.lower','ci.upper','std.all')])

cat('\n8) Invariancia multigrupo\n')
fit_config <- sem(partial_model, data=d, group='GENERO', estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_metric <- sem(partial_model, data=d, group='GENERO', group.equal=c('loadings'), estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_metric_partial <- sem(partial_model, data=d, group='GENERO', group.equal=c('loadings'), group.partial=c('CO=~CO3'), estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_scalar <- sem(partial_model, data=d, group='GENERO', group.equal=c('loadings','intercepts'), group.partial=c('CO=~CO3'), estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fit_struct <- sem(partial_model, data=d, group='GENERO', group.equal=c('loadings','regressions'), group.partial=c('CO=~CO3'), estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
fits <- list(Configural=fit_config, Metrica=fit_metric, Metrica_parcial=fit_metric_partial, Escalar=fit_scalar, Estructural=fit_struct)
for (nm in names(fits)) {
  cat('\n', nm, '\n')
  print(round(fitMeasures(fits[[nm]], c('chisq.scaled','df.scaled','pvalue.scaled','cfi.robust','tli.robust','rmsea.robust','srmr','aic','bic')), 4))
}
cat('\nDiferencias robustas\n')
print(lavTestLRT(fit_config, fit_metric, method='satorra.bentler.2010'))
print(lavTestLRT(fit_config, fit_metric_partial, method='satorra.bentler.2010'))
print(lavTestLRT(fit_metric_partial, fit_scalar, method='satorra.bentler.2010'))
print(lavTestLRT(fit_metric_partial, fit_struct, method='satorra.bentler.2010'))

cat('\n9) Diferencias específicas por género\n')
pec <- parameterEstimates(fit_config, standardized=TRUE, ci=TRUE)
reg <- pec[pec$op=='~',]
for (path in unique(paste(reg$lhs, reg$rhs, sep='~'))) {
  lhs <- sub('~.*','',path); rhs <- sub('.*~','',path)
  r1 <- reg[reg$lhs==lhs & reg$rhs==rhs & reg$group==1,]
  r2 <- reg[reg$lhs==lhs & reg$rhs==rhs & reg$group==2,]
  diff <- r1$est-r2$est; se <- sqrt(r1$se^2 + r2$se^2); z <- diff/se; pval <- 2*pnorm(abs(z), lower.tail=FALSE)
  cat(path, 'hombres=',round(r1$est,4),'mujeres=',round(r2$est,4),'diff=',round(diff,4),'z=',round(z,4),'p=',round(pval,4),'\n')
}
'''
Path('pregunta3_lavaan.R').write_text(r_code)
print('Script R creado: pregunta3_lavaan.R')


Script R creado: pregunta3_lavaan.R


In [5]:
import subprocess, sys
resultado = subprocess.run(['Rscript', 'pregunta3_lavaan.R'], text=True, capture_output=True, timeout=1200)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
    raise SystemExit(resultado.returncode)


Base usada: ../data/Laboral.sav 
N total: 400 
Frecuencia GENERO: 0=Masculino, 1=Femenino

  0   1 
200 200 
Valores perdidos totales: 0 

1) CFA robusto
 chisq.scaled     df.scaled pvalue.scaled    cfi.robust    tli.robust 
      216.902       160.000         0.002         0.987         0.985 
 rmsea.robust          srmr           aic           bic 
        0.028         0.033     24647.543     24926.945 

2) Cargas factoriales CFA
   lhs rhs   est    se      z pvalue ci.lower ci.upper std.all
1   AC AC1 1.144 0.045 25.321      0    1.056    1.233   0.822
2   AC AC2 1.414 0.053 26.878      0    1.311    1.517   0.820
3   AC AC3 1.187 0.043 27.559      0    1.102    1.271   0.837
4   AC AC4 1.312 0.053 24.744      0    1.208    1.416   0.815
5   PA PA1 1.265 0.116 10.869      0    1.037    1.493   0.692
6   PA PA2 1.307 0.101 12.947      0    1.109    1.505   0.804
7   PA PA3 1.038 0.078 13.367      0    0.886    1.190   0.778
8   PA PA4 1.156 0.084 13.814      0    0.992    1.320   0.

## Inspección secuencial de cargas para la invariancia métrica parcial

El documento reporta que la invariancia métrica completa queda levemente fuera del criterio práctico y que se identificó la carga de `CO3` mediante inspección secuencial. Esta celda documenta ese procedimiento: se libera **una carga a la vez** sobre el modelo métrico y se compara cada solución contra el configural. La carga cuya liberación produce la mayor recuperación de ajuste (menor $\Delta\chi^2$ escalado y $\Delta CFI$ dentro del criterio) es la candidata a invariancia parcial.

In [7]:
from pathlib import Path
r_seq = r"""
ns <- asNamespace('parallel'); unlockBinding('detectCores', ns)
assign('detectCores', function(all.tests=FALSE, logical=TRUE) 8L, envir=ns); lockBinding('detectCores', ns)
library(lavaan); library(haven)

find_data <- function() {
  candidates <- c('../data/Laboral.sav','../../data/Laboral.sav','Tarea2/data/Laboral.sav','data/Laboral.sav','Laboral.sav')
  for (f in candidates) if (file.exists(f)) return(f)
  download.file('https://raw.githubusercontent.com/sebabecerra/Metodos-Cuantitativos-DEN/main/Tarea2/data/Laboral.sav','Laboral.sav',mode='wb',quiet=TRUE)
  'Laboral.sav'
}
d <- as.data.frame(read_sav(find_data()))
vars <- c(paste0('AC',1:4),paste0('PA',1:4),paste0('ST',1:4),paste0('CO',1:4),paste0('IP',1:4),'GENERO')
d <- d[, vars]; d$GENERO <- as.integer(d$GENERO)

measurement <- '
AC =~ AC1 + AC2 + AC3 + AC4
PA =~ PA1 + PA2 + PA3 + PA4
ST =~ ST1 + ST2 + ST3 + ST4
CO =~ CO1 + CO2 + CO3 + CO4
IP =~ IP1 + IP2 + IP3 + IP4
'
partial_model <- paste0(measurement, '
ST ~ PA + AC
CO ~ PA + AC + ST
IP ~ PA + AC + ST + CO
')

fit_config <- sem(partial_model, data=d, group='GENERO', estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1)
cfi_config <- fitMeasures(fit_config, 'cfi.robust')

items <- c(paste0('AC',1:4),paste0('PA',1:4),paste0('ST',1:4),paste0('CO',1:4),paste0('IP',1:4))
factores <- rep(c('AC','PA','ST','CO','IP'), each=4)

cat('Liberacion de una carga a la vez sobre el modelo metrico (comparado contra configural):\n')
cat(sprintf('%-10s %12s %10s %10s\n','Carga libre','Dchi2_SB(14)','p','DCFI'))
resultados <- data.frame()
for (i in seq_along(items)) {
  etiqueta <- paste0(factores[i], '=~', items[i])
  fit_i <- try(sem(partial_model, data=d, group='GENERO', group.equal='loadings',
                   group.partial=etiqueta, estimator='MLR', missing='fiml', std.lv=TRUE, ncpus=1), silent=TRUE)
  if (inherits(fit_i, 'try-error')) next
  lrt <- lavTestLRT(fit_config, fit_i, method='satorra.bentler.2010')
  dcfi <- fitMeasures(fit_i,'cfi.robust') - cfi_config
  cat(sprintf('%-10s %12.3f %10.4g %10.4f\n', etiqueta, lrt$`Chisq diff`[2], lrt$`Pr(>Chisq)`[2], dcfi))
  resultados <- rbind(resultados, data.frame(carga=etiqueta, dchi2=lrt$`Chisq diff`[2], p=lrt$`Pr(>Chisq)`[2], dcfi=dcfi))
}
mejor <- resultados[which.max(resultados$dcfi),]
cat('\nMayor recuperacion de ajuste al liberar:', mejor$carga, '\n')
"""
Path('pregunta3_invarianza_secuencial.R').write_text(r_seq)
import subprocess
res = subprocess.run(['Rscript','pregunta3_invarianza_secuencial.R'], text=True, capture_output=True, timeout=1800)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr); raise SystemExit(res.returncode)

Liberacion de una carga a la vez sobre el modelo metrico (comparado contra configural):
Carga libre Dchi2_SB(14)          p       DCFI
AC=~AC1          33.607   0.002353    -0.0081
AC=~AC2          30.059    0.00749    -0.0070
AC=~AC3          41.003  0.0001777    -0.0109
AC=~AC4          41.111  0.0001709    -0.0111
PA=~PA1          41.226  0.0001639    -0.0110
PA=~PA2          40.339  0.0002258    -0.0103
PA=~PA3          41.481  0.0001494    -0.0109
PA=~PA4          41.550  0.0001456    -0.0112
ST=~ST1          40.708  0.0001977    -0.0106
ST=~ST2          41.630  0.0001415    -0.0112
ST=~ST3          37.695  0.0005785    -0.0101
ST=~ST4          41.517  0.0001474    -0.0112
CO=~CO1          39.249  0.0003338    -0.0103
CO=~CO2          38.065  0.0005079    -0.0094
CO=~CO3          30.968   0.005601    -0.0058
CO=~CO4          46.043  2.759e-05    -0.0116
IP=~IP1          39.932  0.0002614    -0.0106
IP=~IP2          41.869  0.0001297    -0.0111
IP=~IP3          40.981  0.0001791   

## Conclusión operativa

El modelo de medición presenta buen ajuste, confiabilidad y validez. El modelo parcialmente mediado representa mejor los datos que el completamente mediado. Los efectos indirectos apoyados son los que pasan por compromiso organizacional. En el análisis multigrupo se obtiene invariancia métrica parcial, no se comparan medias latentes por falta de invariancia escalar, y los caminos estructurales muestran comparabilidad aproximada con una posible diferencia localizada en `AC -> ST`.
